In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install(
    'transformers<5.0.0',
    'timm>=0.9.12',
    'scikit-learn',
    'torchmetrics',
)
print('All packages installed.')


In [ ]:
#we need to restart the kernel, after upgrade protobuf
%pip install --upgrade protobuf
import google.protobuf
print(google.protobuf.__version__)


In [ ]:
%pip install torch>=2.6.0+cu124

In [1]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('causal-conv1d>=1.4.0', 'mamba-ssm>=2.2.0', '--no-build-isolation')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 23.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
#! pip install mambavision==1.1.0

In [2]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, time, copy, warnings, random
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image

import timm
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy

from transformers import AutoModelForImageClassification

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
from torchmetrics import (
    Accuracy, Precision, Recall, F1Score,
    AUROC, AveragePrecision, CohenKappa, MatthewsCorrCoef
)

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# ── Determinism ───────────────────────────────────────────────────────────────
SEED = 42
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f' Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')



In [ ]:
#!pip3 install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
#!pip install torch --index-url https://lnkd.in/g_8dBghS

# 1. Clear out any previous partial or broken installations
#!pip uninstall -y mamba-ssm causal-conv1d

# 2. Install the required causal-conv1d dependency without build isolation
#!pip install causal-conv1d>=1.4.0 --no-build-isolation

# 3. Install the core mamba package without build isolation
#!pip install mamba-ssm --no-build-isolation

In [ ]:
from mamba_ssm import Mamba
print("✅ Mamba loaded successfully")

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from mamba_ssm import Mamba  # Specialized linear-time state-space layer

# 1. DEFINE THE HYBRID ARCHITECTURE
class HybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, layer_type="mamba"):
        super().__init__()
        self.layer_type = layer_type
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        if layer_type == "mamba":
            self.core_layer = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
        else:
            self.core_layer = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
            
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        # Layer 1: Core Attention / SSM processing with residual connection
        if self.layer_type == "mamba":
            x = x + self.core_layer(self.norm1(x))
        else:
            norm_x = self.norm1(x)
            attn_out, _ = self.core_layer(norm_x, norm_x, norm_x)
            x = x + attn_out
            
        # Layer 2: Feed Forward network with residual connection
        x = x + self.ffn(self.norm2(x))
        return x

class HybridSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8, num_layers=6, ratio=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.layers = nn.ModuleList()
        
        # Configure a 3:1 Mamba-to-Attention layer pattern layout
        for idx in range(num_layers):
            layer_type = "attention" if (idx + 1) % (ratio + 1) == 0 else "mamba"
            self.layers.append(HybridBlock(embed_dim, num_heads, layer_type))
            
        self.pooler = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.pooler(x.transpose(1, 2)).squeeze(-1)
        return self.classifier(x)

# 2. DATA PREPARATION WORKFLOW
def prepare_data(batch_size=32, max_length=256):
    print("Loading IMDb dataset...")
    dataset = load_dataset("stanfordnlp/imdb")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
    print("Tokenizing textual reviews...")
    tokenized_datasets = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_datasets.set_format("torch")
    
    # Generate structured dataloaders
    train_loader = DataLoader(tokenized_datasets["train"], batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(tokenized_datasets["test"], batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, tokenizer.vocab_size

# 3. TRAINING & VALIDATION LOOPS
def train_and_evaluate():
    # Setup Device & parameters
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running pipeline on: {device}")
    
    max_review_len = 256
    train_loader, test_loader, vocab_size = prepare_data(batch_size=32, max_length=max_review_len)
    
    # Initialize the Hybrid Classifier model
    model = HybridSentimentClassifier(vocab_size=vocab_size, num_layers=8, ratio=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    
    epochs = 5
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * input_ids.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        train_acc = (correct / total) * 100
        avg_loss = total_loss / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Train Accuracy: {train_acc:.2f}%")
        
        # Test validation evaluation phase at each epoch end
        evaluate_test_set(model, test_loader, device)

def evaluate_test_set(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    test_acc = (correct / total) * 100
    print(f">> Final Test Accuracy on 25k validation reviews: {test_acc:.2f}%\n")

if __name__ == "__main__":
    train_and_evaluate()


Running pipeline on: cuda
Loading IMDb dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Tokenizing textual reviews...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Epoch 1/5 | Loss: 0.6358 | Train Accuracy: 62.20%
>> Final Test Accuracy on 25k validation reviews: 71.98%

Epoch 2/5 | Loss: 0.4772 | Train Accuracy: 77.00%
>> Final Test Accuracy on 25k validation reviews: 76.54%

Epoch 3/5 | Loss: 0.4118 | Train Accuracy: 81.25%
>> Final Test Accuracy on 25k validation reviews: 81.44%

Epoch 4/5 | Loss: 0.3725 | Train Accuracy: 83.47%
>> Final Test Accuracy on 25k validation reviews: 82.66%

Epoch 5/5 | Loss: 0.3362 | Train Accuracy: 85.32%
>> Final Test Accuracy on 25k validation reviews: 82.54%



In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from mamba_ssm import Mamba

# 1. ATTENTIVE POOLING BLOCK
class AttentivePooling(nn.Module):
    """
    Learns dynamic attention weights over sequence tokens 
    instead of treating them uniformly like average pooling.
    """
    def __init__(self, embed_dim):
        super().__init__()
        self.attn_weights = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        weights = torch.softmax(self.attn_weights(x), dim=1)
        return torch.sum(x * weights, dim=1)  # Output: [batch_size, embed_dim]

# 2. INTERLEAVED HYBRID LAYERS
class HybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, layer_type="mamba"):
        super().__init__()
        self.layer_type = layer_type
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        if layer_type == "mamba":
            self.core_layer = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
        else:
            self.core_layer = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
            
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        if self.layer_type == "mamba":
            x = x + self.core_layer(self.norm1(x))
        else:
            norm_x = self.norm1(x)
            attn_out, _ = self.core_layer(norm_x, norm_x, norm_x)
            x = x + attn_out
            
        x = x + self.ffn(self.norm2(x))
        return x

class AdvancedHybridClassifier(nn.Module):
    def __init__(self, pretrained_model_name="bert-base-uncased", num_layers=6, ratio=2):
        super().__init__()
        # Load stable pre-trained vocabulary representations
        from transformers import AutoModel
        base_transformer = AutoModel.from_pretrained(pretrained_model_name)
        self.embedding = base_transformer.embeddings
        embed_dim = base_transformer.config.hidden_size  # Automatically matches 768
        
        self.layers = nn.ModuleList()
        for idx in range(num_layers):
            layer_type = "attention" if (idx + 1) % (ratio + 1) == 0 else "mamba"
            self.layers.append(HybridBlock(embed_dim, num_heads=8, layer_type=layer_type))
            
        self.pooler = AttentivePooling(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids):
        # input_ids shape: [batch_size, sequence_length]
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.pooler(x)
        return self.classifier(x)

# 3. HIGH-CAPACITY DATA PIPELINE
def get_dataloaders(batch_size=16, max_length=512):
    print("Fetching IMDb benchmark data...")
    dataset = load_dataset("stanfordnlp/imdb")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
    print("Encoding sequences (Max context depth: 512)...")
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized.set_format("torch")
    
    train_loader = DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(tokenized["test"], batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# 4. ROBUST TRAIN LOOP WITH WARMUP SCHEDULING
def execute_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Active runtime acceleration context: {device}")
    
    # Utilizing an optimization batch configuration matching standard sizing
    train_loader, test_loader = get_dataloaders(batch_size=16, max_length=512)
    
    model = AdvancedHybridClassifier().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
    
    epochs = 4
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=int(0.1 * total_steps), 
        num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Prevent gradient explosions
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item() * input_ids.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/total:.4f} | Train Acc: {(correct/total)*100:.2f}%")
        
        # Continuous tracking over test validation set
        evaluate(model, test_loader, device)

def evaluate(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    print(f">> Current Test Set Target Accuracy: {(correct/total)*100:.2f}%\n")

if __name__ == "__main__":
    execute_training()


Active runtime acceleration context: cuda
Fetching IMDb benchmark data...
Encoding sequences (Max context depth: 512)...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1/4 | Loss: 0.4547 | Train Acc: 77.20%
>> Current Test Set Target Accuracy: 86.94%

Epoch 2/4 | Loss: 0.2591 | Train Acc: 90.23%
>> Current Test Set Target Accuracy: 89.13%

Epoch 3/4 | Loss: 0.1582 | Train Acc: 94.81%
>> Current Test Set Target Accuracy: 88.57%

Epoch 4/4 | Loss: 0.0976 | Train Acc: 97.29%
>> Current Test Set Target Accuracy: 88.70%



In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from mamba_ssm import Mamba

# 1. SPECIALIZED ATTENTIVE POOLING BLOCK
class AttentivePooling(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attn_weights = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        weights = torch.softmax(self.attn_weights(x), dim=1)
        return torch.sum(x * weights, dim=1)  # Output: [batch_size, embed_dim]

# 2. INTERLEAVED HYBRID BLOCK ARCHITECTURE
class HybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, layer_type="mamba"):
        super().__init__()
        self.layer_type = layer_type
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        if layer_type == "mamba":
            self.core_layer = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
        else:
            self.core_layer = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
            
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        if self.layer_type == "mamba":
            x = x + self.core_layer(self.norm1(x))
        else:
            norm_x = self.norm1(x)
            attn_out, _ = self.core_layer(norm_x, norm_x, norm_x)
            x = x + attn_out
            
        x = x + self.ffn(self.norm2(x))
        return x

class ContextualHybridClassifier(nn.Module):
    def __init__(self, pretrained_model_name="distilbert-base-uncased", num_layers=4, ratio=1):
        super().__init__()
        # Load full contextual encoder block
        self.encoder_backbone = AutoModel.from_pretrained(pretrained_model_name)
        
        # Freeze encoder backbone weights to prevent overfitting on 50k dataset
        for param in self.encoder_backbone.parameters():
            param.requires_grad = False
            
        embed_dim = self.encoder_backbone.config.hidden_size  # Automatically extracts 768
        
        # Setting up hybrid layers to parse deep contextualized embeddings
        self.layers = nn.ModuleList()
        for idx in range(num_layers):
            layer_type = "attention" if (idx + 1) % (ratio + 1) == 0 else "mamba"
            self.layers.append(HybridBlock(embed_dim, num_heads=8, layer_type=layer_type))
            
        self.pooler = AttentivePooling(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),  # Increased dropout to explicitly target overfitting
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        # Extract rich contextual sequences from frozen backbone
        with torch.no_grad():
            encoder_outputs = self.encoder_backbone(input_ids=input_ids, attention_mask=attention_mask)
            x = encoder_outputs.last_hidden_state  # Shape: [batch_size, seq_len, 768]
            
        # Process context signals through custom hybrid blocks
        for layer in self.layers:
            x = layer(x)
            
        x = self.pooler(x)
        return self.classifier(x)

# 3. SEQUENCE DATA PIPELINE
def get_dataloaders(batch_size=32, max_length=256):
    print("Fetching IMDb benchmark data...")
    dataset = load_dataset("stanfordnlp/imdb")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    
    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
    print("Extracting tokenizer maps...")
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    
    train_loader = DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(tokenized["test"], batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# 4. ENGINE RUNNER
def execute_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Active hardware context: {device}")
    
    train_loader, test_loader = get_dataloaders(batch_size=32, max_length=256)
    
    model = ContextualHybridClassifier().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.02)
    
    epochs = 3
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item() * input_ids.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/total:.4f} | Train Acc: {(correct/total)*100:.2f}%")
        evaluate(model, test_loader, device)

def evaluate(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids, attention_mask)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    print(f">> Adjusted Validation Target Accuracy: {(correct/total)*100:.2f}%\n")

if __name__ == "__main__":
    execute_training()


Active hardware context: cuda
Fetching IMDb benchmark data...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Epoch 1/3 | Loss: 0.3575 | Train Acc: 84.36%
>> Adjusted Validation Target Accuracy: 88.57%

Epoch 2/3 | Loss: 0.2501 | Train Acc: 90.16%
>> Adjusted Validation Target Accuracy: 90.37%

Epoch 3/3 | Loss: 0.1869 | Train Acc: 93.07%
>> Adjusted Validation Target Accuracy: 90.58%



In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from mamba_ssm import Mamba

# 1. OPTIMIZED ATTENTIVE POOLING
class AttentivePooling(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attn_weights = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        weights = torch.softmax(self.attn_weights(x), dim=1)
        return torch.sum(x * weights, dim=1)

# 2. INTERLEAVED HYBRID LAYERS
class HybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, layer_type="mamba"):
        super().__init__()
        self.layer_type = layer_type
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        if layer_type == "mamba":
            self.core_layer = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
        else:
            self.core_layer = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
            
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        if self.layer_type == "mamba":
            x = x + self.core_layer(self.norm1(x))
        else:
            norm_x = self.norm1(x)
            attn_out, _ = self.core_layer(norm_x, norm_x, norm_x)
            x = x + attn_out
            
        x = x + self.ffn(self.norm2(x))
        return x

class FineTunedHybridClassifier(nn.Module):
    def __init__(self, pretrained_model_name="distilbert-base-uncased", num_layers=4, ratio=1):
        super().__init__()
        self.encoder_backbone = AutoModel.from_pretrained(pretrained_model_name)
        
        # Freeze ALL layers initially
        for param in self.encoder_backbone.parameters():
            param.requires_grad = False
            
        # CRITICAL UNFREEZE: Open up the final transformer layer of DistilBERT for fine-tuning
        # DistilBERT has 6 transformer layers (0 to 5) located inside '.transformer.layer'
        for param in self.encoder_backbone.transformer.layer[-1].parameters():
            param.requires_grad = True
            
        embed_dim = self.encoder_backbone.config.hidden_size  # 768
        
        self.layers = nn.ModuleList()
        for idx in range(num_layers):
            layer_type = "attention" if (idx + 1) % (ratio + 1) == 0 else "mamba"
            self.layers.append(HybridBlock(embed_dim, num_heads=8, layer_type=layer_type))
            
        self.pooler = AttentivePooling(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),  # Strong dropout to combat overfitting
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        # Pass tokens through the backbone (allowing gradients for the top layer only)
        encoder_outputs = self.encoder_backbone(input_ids=input_ids, attention_mask=attention_mask)
        x = encoder_outputs.last_hidden_state  
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.pooler(x)
        return self.classifier(x)

# 3. SEQUENCE PIPELINE (Increased max_length slightly to capture more trailing sentiment context)
def get_dataloaders(batch_size=32, max_length=320):
    print("Fetching IMDb dataset...")
    dataset = load_dataset("stanfordnlp/imdb")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    
    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
    print("Encoding sequences...")
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    
    train_loader = DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(tokenized["test"], batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# 4. DISCRIMINATIVE OPTIMIZER ENGINE
def execute_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Active hardware context: {device}")
    
    train_loader, test_loader = get_dataloaders(batch_size=32, max_length=320)
    model = FineTunedHybridClassifier().to(device)
    
    # Separate parameters for Layer-Wise Learning Rates (LLRD)
    backbone_params = [p for n, p in model.encoder_backbone.named_parameters() if p.requires_grad]
    custom_layers_params = [p for n, p in model.named_parameters() if not n.startswith('encoder_backbone') and p.requires_grad]
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5},       # Very gentle updates for the pre-trained encoder layer
        {'params': custom_layers_params, 'lr': 5e-5}   # Faster rate for custom hybrid blocks
    ], weight_decay=0.05)                             # High weight decay targets overfitting
    
    epochs = 3
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item() * input_ids.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/total:.4f} | Train Acc: {(correct/total)*100:.2f}%")
        evaluate(model, test_loader, device)

def evaluate(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids, attention_mask)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    print(f">> Adjusted Validation Target Accuracy: {(correct/total)*100:.2f}%\n")

if __name__ == "__main__":
    execute_training()


Active hardware context: cuda
Fetching IMDb dataset...
Encoding sequences...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Epoch 1/3 | Loss: 0.3489 | Train Acc: 84.45%
>> Adjusted Validation Target Accuracy: 89.62%

Epoch 2/3 | Loss: 0.2396 | Train Acc: 90.51%
>> Adjusted Validation Target Accuracy: 91.17%

Epoch 3/3 | Loss: 0.1903 | Train Acc: 92.78%
>> Adjusted Validation Target Accuracy: 91.27%



In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from mamba_ssm import Mamba

# 1. OPTIMIZED ATTENTIVE POOLING
class AttentivePooling(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attn_weights = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        weights = torch.softmax(self.attn_weights(x), dim=1)
        return torch.sum(x * weights, dim=1)

# 2. INTERLEAVED HYBRID LAYERS
class HybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, layer_type="mamba"):
        super().__init__()
        self.layer_type = layer_type
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        if layer_type == "mamba":
            self.core_layer = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
        else:
            self.core_layer = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
            
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, x):
        if self.layer_type == "mamba":
            x = x + self.core_layer(self.norm1(x))
        else:
            norm_x = self.norm1(x)
            attn_out, _ = self.core_layer(norm_x, norm_x, norm_x)
            x = x + attn_out
            
        x = x + self.ffn(self.norm2(x))
        return x

class ProductionHybridClassifier(nn.Module):
    def __init__(self, pretrained_model_name="distilbert-base-uncased", num_layers=4, ratio=1):
        super().__init__()
        self.encoder_backbone = AutoModel.from_pretrained(pretrained_model_name)
        
        # Freeze lower layers
        for param in self.encoder_backbone.parameters():
            param.requires_grad = False
            
        # Keep the final transformer layer un-frozen for domain adaptation
        for param in self.encoder_backbone.transformer.layer[-1].parameters():
            param.requires_grad = True
            
        embed_dim = self.encoder_backbone.config.hidden_size  # 768
        
        self.layers = nn.ModuleList()
        for idx in range(num_layers):
            layer_type = "attention" if (idx + 1) % (ratio + 1) == 0 else "mamba"
            self.layers.append(HybridBlock(embed_dim, num_heads=8, layer_type=layer_type))
            
        self.pooler = AttentivePooling(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.encoder_backbone(input_ids=input_ids, attention_mask=attention_mask)
        x = encoder_outputs.last_hidden_state  
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.pooler(x)
        return self.classifier(x)

# 3. FULL CONTEXT SEQUENCE PIPELINE (MAX CAPACITY: 512)
def get_dataloaders(batch_size=32, max_length=512):
    print("Fetching IMDb dataset...")
    dataset = load_dataset("stanfordnlp/imdb")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    
    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
    
    print("Encoding sequences up to 512 tokens...")
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    
    train_loader = DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(tokenized["test"], batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# 4. RUNNER ENGINE
def execute_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Active hardware context: {device}")
    
    # 512 token limits maximize context capture
    train_loader, test_loader = get_dataloaders(batch_size=32, max_length=512)
    model = ProductionHybridClassifier().to(device)
    
    backbone_params = [p for n, p in model.encoder_backbone.named_parameters() if p.requires_grad]
    custom_layers_params = [p for n, p in model.named_parameters() if not n.startswith('encoder_backbone') and p.requires_grad]
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5},       
        {'params': custom_layers_params, 'lr': 5e-5}   
    ], weight_decay=0.05)                             
    
    epochs = 3
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item() * input_ids.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/total:.4f} | Train Acc: {(correct/total)*100:.2f}%")
        evaluate(model, test_loader, device)

def evaluate(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids, attention_mask)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    print(f">> Adjusted Validation Target Accuracy: {(correct/total)*100:.2f}%\n")

if __name__ == "__main__":
    execute_training()


Active hardware context: cuda
Fetching IMDb dataset...
Encoding sequences up to 512 tokens...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Epoch 1/3 | Loss: 0.3417 | Train Acc: 85.05%
>> Adjusted Validation Target Accuracy: 90.34%

Epoch 2/3 | Loss: 0.2238 | Train Acc: 91.34%
>> Adjusted Validation Target Accuracy: 92.23%

Epoch 3/3 | Loss: 0.1769 | Train Acc: 93.36%
>> Adjusted Validation Target Accuracy: 92.39%

